In [16]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np


In [17]:
df_base_Recibida = pd.read_excel(r'C:\Users\alejandro_arboledas\Downloads\Usuarios sin inicio de sesión 02_02_2025.xlsx')
df_base_Recibida

,Correo,Nombre,Documento,Telefono,Email,Fecha_de_creacion,Ultimo_Inicio_Sesion,Esta_Suspendido,Esta_Archivado,Unidad_Organizativa,Titulo,Departamento,Email_Recuperacion,Telefono_Recuperacion
0,abraham.fernandez@cun.edu.co,ABRAHAM FARITH FERNANDEZ AGUIRRE,1083006478,NaN,abraham.fernandez@cun.edu.co,2025-08-15T20:45:36.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/ESCUELA DE CIENCIAS ADMINISTRATIVAS,ESTUDIANTE,NaN,NaN,NaN
1,adalbert.camargo@cun.edu.co,Adalbert Nicolas Camargo Reyes,1122512721,+573246246218,adalbert.camargo@cun.edu.co,2026-01-30T00:03:26.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/POR DEFINIR,ESTUDIANTE,POR DEFINIR,adalnico04@gmail.com,5.732462e+11
2,adela.ceballos@cun.edu.co,ADELA SOFIA CEBALLOS TORDECILLA,1067851665,NaN,adeceballos.12@gmail.com,2025-08-06T19:57:22.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/ESCUELA DE CIENCIAS ADMINISTRATIVAS,ESTUDIANTE,ADMINISTRACIÓN DE EMPRESAS,adeceballos.12@gmail.com,NaN
3,adelaida.vargas@cun.edu.co,ADELAIDA YURLETH VARGAS SUAZA,1019105326,+573162497306,adelaida.vargas@cun.edu.co,2026-01-31T00:43:07.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/SOCIO HUMANISTICA,ESTUDIANTE,SOCIO HUMANISTICA,adelaida.vargas.suaza@gmail.com,5.731625e+11
4,adolfo.miranda@cun.edu.co,ADOLFO DEL MIRANDA GONZALEZ,1067953805,+573005646815,adolfo.miranda@cun.edu.co,2026-01-27T04:52:23.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/SOCIO HUMANISTICA,ESTUDIANTE,SOCIO HUMANISTICA,adomigo9@hotmail.com,5.730056e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3962,zuli.medina@cun.edu.co,Zuli Dayanna Medina Leon,1027402383,+573204110199,zuli.medina@cun.edu.co,2026-01-31T16:17:51.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/POR DEFINIR,ESTUDIANTE,POR DEFINIR,zulymedinaleon18@gmail.com,5.732041e+11
3963,zully.yasno@cun.edu.co,ZULLY MARYOLY YASNO HOYOS,55130300,+573113156174,zully.yasno@cun.edu.co,2025-12-30T23:32:30.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/ESCUELA DE COMUNICACION Y BELLAS ...,ESTUDIANTE,ESCUELA DE COMUNICACION Y BELLAS ARTES,zullyyasno03@gmail.com,5.731132e+11
3964,zulma.cano@cun.edu.co,Zulma Lisset Cano Rodriguez,53011569,+573209831239,zulma.cano@cun.edu.co,2026-01-31T15:27:36.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/POR DEFINIR,ESTUDIANTE,POR DEFINIR,litzcano@gmail.com,5.732098e+11
3965,zulma.trivino@cun.edu.co,ZULMA EGLETH TRIVIÑO GONZÁLEZ,52663033,+573133130634,sofysromerot@gmail.com,2025-10-01T13:41:39.000Z,1970-01-01T00:00:00.000Z,False,False,/ESTUDIANTES/ESCUELA DE CIENCIAS ADMINISTRATIVAS,ESTUDIANTE,ADMINISTRACION PUBLICA,sofysromerot@gmail.com,5.731331e+11


In [18]:
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [19]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT PERIODO FROM CUN.MATRIZ_CALENDARIO
                    WHERE Estado_convocatoria = 'ACTIVO'
        
            """
            PERIODOS_ACTIVOS = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

In [20]:
para_orden = df_base_Recibida['Correo'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
periodos_unicos = PERIODOS_ACTIVOS['PERIODO'].unique()
valores_unicos_periodos = "', '".join(map(str, periodos_unicos))
valores_unicos_periodos = f"'{valores_unicos_periodos}'"

In [21]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT distinct(DOC_ALUM),CONCAT(NOM_TERCERO,' ',PRI_APELLIDO, ' ',SEG_APELLIDO) AS NOMBRE ,EMAIL_INS AS 'EMAIL INSTITUCIONAL',TEL_CELULAR,TEL_CASA,DIRECCION_CASA,EMAIL_PER  FROM ZOHO.BASE_PERSONAS  
                    WHERE EMAIL_INS IN ({valores_unicos_df1_orden})
                    and estado_pago = 'pago'
                    and est_matriculado = '1-activo'
                    AND PERIODO IN ({valores_unicos_periodos})
        
            """
            df_base_persona = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

In [22]:
df_base_persona['TEL_CELULAR'] = df_base_persona['TEL_CELULAR'].fillna(df_base_persona['TEL_CASA'])
df_base_persona

,DOC_ALUM,NOMBRE,EMAIL INSTITUCIONAL,TEL_CELULAR,TEL_CASA,DIRECCION_CASA,EMAIL_PER
0,1005933017,ORLANDO PINZÓN REINOSO,orlando.pinzon@cun.edu.co,3102965365,3102965365,----,opinzonr@ut.edu.co
1,1010165366,JANIS GÓMEZ PINEDA,janis.gomez@cun.edu.co,3214478820,3214478820,CL 65 N # 17 64,janissofigomez@hotmail.com
2,1014197763,MARÍA MARTÍNEZ LEÓN,maria.martinezleo@cun.edu.co,3505164082,3505164082,TR 75 A 82 H 54,mafemale6@gmail.com
3,1017178230,YULITZA BLANCO DORADO,yulitza.blanco@cun.edu.co,3045532903,3045532903,barrrio el bosque,yulitza.blanco.dorado@gmail.com
4,1018448159,YISETH GUALTEROS FERRO,yiseth.gualteros@cun.edu.co,3143224807,3143224807,KR 9 A 8 40,gualterosleyes@outlook.es
5,1024531083,MIRYEN DELGADO DAVID,miryen.delgado@cun.edu.co,573013570226,573013570226,calle 79 b 42 15,yen.1992@hotmail.com
6,1024575942,LADY RAMÍREZ AGUDELO,lady.ramirezagu@cun.edu.co,3102816781,3102816781,calle 57 z bis A sur # 77 c 16,lady13ramirez@gmail.com
7,1026265031,JENIFFER VARGAS CORREA,jeniffer.vargas@cun.edu.co,3008366925,3008366925,CL 53 SUR 79 C 21,jevacof@gmail.com
8,1032401306,JAVIER MARTINEZ BAEZ,javier.martinezbae@cun.edu.co,3233232839,3233232839,CL 146 73 A 20,ezbaorlan179@gmail.com
9,1057410537,LADY CUBIDES SUAREZ,lady.cubidess@cun.edu.co,3112861182,3112861182,KR 70 C 1 72,lpaola1703@gmail.com


In [23]:
df_base_persona = df_base_persona.drop_duplicates()
df_base_persona.shape

(49, 7)

In [24]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT DIR_EMAIL FROM dwh_Inicio_clases
                    WHERE EstadoMoodle = 'Sin registro'
                    and PERIODO IN ({valores_unicos_periodos})
        
            """
            df_moodle = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()
df_moodle

,DIR_EMAIL
0,alison.cifuentes@cun.edu.co
1,laura.fajardorod@cun.edu.co
2,nicoll.mafla@cun.edu.co
3,orlando.pinzon@cun.edu.co
4,janis.gomez@cun.edu.co
...,...
113,ingmanuelcb@gmail.com
114,johny.fontecha@cun.edu.co
115,edwar.savio@cun.edu.co
116,carlos.condeg@cun.edu.co


In [25]:
para_orden = df_moodle['DIR_EMAIL'].unique()
valores_unicos = "', '".join(map(str, para_orden))
valores_unicos = f"'{valores_unicos}'"

In [26]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT distinct(DOC_ALUM),CONCAT(NOM_TERCERO,' ',PRI_APELLIDO, ' ',SEG_APELLIDO) AS NOMBRE ,EMAIL_INS AS 'EMAIL INSTITUCIONAL',TEL_CELULAR,TEL_CASA,DIRECCION_CASA,EMAIL_PER  FROM ZOHO.BASE_PERSONAS  
                    WHERE EMAIL_INS IN ({valores_unicos})
                    and estado_pago = 'pago'
                    and est_matriculado = '1-activo'
                    AND PERIODO IN ({valores_unicos_periodos})
        
            """
            df_base_persona_moodle = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

In [27]:
df_base_persona_moodle['TEL_CELULAR'] = df_base_persona_moodle['TEL_CELULAR'].fillna(df_base_persona_moodle['TEL_CASA'])
df_base_persona_moodle.shape

(118, 7)

In [28]:
df_base_persona_moodle = df_base_persona_moodle.drop_duplicates()
df_base_persona_moodle

,DOC_ALUM,NOMBRE,EMAIL INSTITUCIONAL,TEL_CELULAR,TEL_CASA,DIRECCION_CASA,EMAIL_PER
0,1000774311,ALISON CIFUENTES VARGAS,alison.cifuentes@cun.edu.co,3143912567,3143912567,KR 5 48 I 32 SUR,natalia.cifuentes015@gmail.com
1,1002526428,LAURA FAJARDO RODRIGUEZ,laura.fajardorod@cun.edu.co,3185008229,3185008229,CL 128 C # 103- 5,fajardolaura794@gmail.com
2,1005754766,NICOLL MAFLA IREGUI,nicoll.mafla@cun.edu.co,3116654748,3116654748,Mz 82 Cs 19 3 Etp Ciudadela Simón Bolivar ibague,nicolldahianamafla@gmail.com
3,1005933017,ORLANDO PINZÓN REINOSO,orlando.pinzon@cun.edu.co,3102965365,3102965365,----,opinzonr@ut.edu.co
4,1010165366,JANIS GÓMEZ PINEDA,janis.gomez@cun.edu.co,3214478820,3214478820,CL 65 N # 17 64,janissofigomez@hotmail.com
...,...,...,...,...,...,...,...
113,80002184,MANUEL CASTRO BALLESTEROS,ingmanuelcb@gmail.com,3046338190,3046338190,CRA 82 H # 62-22,ingmanuelcb@gmail.com
114,80070658,JOHNY FONTECHA ESPITIA,johny.fontecha@cun.edu.co,3229118989,3229118989,KR 7 #3C- 20,arteferneyespitia@gmail.com
115,80578339,EDWAR SAVIO LUGO,edwar.savio@cun.edu.co,3107771166,3107771166,KR 4 20 00,alexandersavio@hotmail.com
116,94556368,CARLOS CONDE GAVIRIA,carlos.condeg@cun.edu.co,3187827852,3187827852,CL 2 A # 8-50 BARRIO LOS LAGOS,carloseduardocondegaviria@gmail.com


In [29]:
##from datetime import datetime
##
##hoy = datetime.now().strftime("%d-%m-%Y")
##print(hoy)
##
##with pd.ExcelWriter(f'BASE_ESTUDIANTES_SINREGISTRO{hoy}.xlsx', engine='openpyxl') as writer:
##    df_base_persona.to_excel(writer, sheet_name='SIN REGISTRO', index=False)
##    df_base_persona_moodle.to_excel(writer, sheet_name='SIN REGISTRO MOODLE', index=False)

In [30]:
df_historico_google = pd.read_excel('BASE_ESTUDIANTES_SINREGISTRO02-02-2026.xlsx', sheet_name='SIN REGISTRO')
df_historico_google.shape

(49, 7)

In [39]:
df_base_persona_final = df_base_persona[~df_base_persona['DOC_ALUM'].isin(df_historico_google['DOC_ALUM'])]
df_base_persona_final.shape

(49, 7)

In [32]:
df_historico_moodle = pd.read_excel('BASE_ESTUDIANTES_SINREGISTRO02-02-2026.xlsx', sheet_name='SIN REGISTRO MOODLE')
df_historico_moodle.shape

(107, 7)

In [35]:
df_base_persona_moodle_final = df_base_persona_moodle[~df_base_persona_moodle['DOC_ALUM'].isin(df_historico_moodle['DOC_ALUM'])]
df_base_persona_moodle_final.shape

(118, 7)

In [36]:
from datetime import datetime

hoy = datetime.now().strftime("%d-%m-%Y")
print(hoy)



03-02-2026


In [37]:
with pd.ExcelWriter(f'BASE_ESTUDIANTES_SINREGISTRO{hoy}.xlsx', engine='openpyxl') as writer:
    df_base_persona_final.to_excel(writer, sheet_name='SIN REGISTRO', index=False)
    df_base_persona_moodle_final.to_excel(writer, sheet_name='SIN REGISTRO MOODLE', index=False)